In [8]:
# This script takes as input a NetCDF file of 
# geopotential height at 500 hPa
# and makes a plot of the Earth, orthographic projection, 
# with the geopotential height at 500 hPa plotted a layer whose height is the geopotential height.


# Load the source data
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os

fileIn = "data/download.nc"
ds = xr.open_dataset(fileIn)
# Extract the relevant variables
Z500_raw = ds["z"]  # Geopotential height at 500 hPa
# Squeeze to remove the pressure level dimension, since we know it's only one level
Z500_raw  = Z500_raw.squeeze("pressure_level")

# Extract latitude and longitude coordinates
lat = ds["latitude"]
lon = ds["longitude"]

# Get the time (valid_time) coordinate
time = ds["valid_time"]

# Copy the first longitude band to the end to make sure the plot is closed
# The Z500 has three dimensions: time, latitude, longitude. 
# We want to append the first longitude band to the end of the longitude dimension, 
# and the corresponding Z500 values to the end of the longitude dimension of Z500.

# DO that without using numpy.



# Coarsen  spatially 
resize = 4  # coarsen to 2 degrees

Z500_coarsened = Z500_raw.coarsen(
    latitude=resize, longitude=resize, boundary='trim').mean()
# coarsen to 1 degree
lat = lat.coarsen(latitude=resize, boundary='trim').mean()
lon = lon.coarsen(longitude=resize, boundary='trim').mean()

# Smooth the data but keep a xr dataset
from scipy.ndimage import gaussian_filter
Z500_coarsened_smoothed = xr.DataArray(
    gaussian_filter(Z500_coarsened, sigma=2),
    dims=Z500_coarsened.dims,
    coords=Z500_coarsened.coords
)
print(Z500_coarsened_smoothed.shape)
print(lon[-10:])




(616, 180, 360)
<xarray.DataArray 'longitude' (longitude: 10)>
array([350.375, 351.375, 352.375, 353.375, 354.375, 355.375, 356.375,
       357.375, 358.375, 359.375])
Coordinates:
  * longitude  (longitude) float64 350.4 351.4 352.4 353.4 ... 357.4 358.4 359.4
Attributes:
    units:          degrees_east
    standard_name:  longitude
    long_name:      longitude


In [ ]:
# Append the first  longitude band to the end of the longitude dimension for plotting
lon_appended = xr.concat([lon, lon.isel(longitude=0)], dim="longitude")
Z500_coarsened_smoothed_appended = xr.concat([Z500_coarsened_smoothed, \
    Z500_coarsened_smoothed.isel(longitude=0)], dim="longitude")

# Define climatology as the time average of the data across all time steps
Z500_climatology = Z500_coarsened_smoothed_appended.mean(axis=0)  # Average over the time dimension

# Define anomaly 
Z500_anomaly = Z500_coarsened_smoothed_appended - Z500_climatology  # Subtract the climatology for the corresponding month
Z500_normalized = Z500_anomaly / Z500_climatology  # Normalize by the climatology to get a relative anomaly

# Get the min and max values of the normalized Z500 for color scaling
Z500_min = np.min(Z500_normalized)
Z500_max = np.max(Z500_normalized)




# Plot a sphere with the height of the layer given by Z500_normalized
# Create a grid of latitude and longitude
lon_grid, lat_grid = np.meshgrid(lon_appended, lat)
# Convert to Cartesian coordinates for plotting
R = 1  # Earth radius in kilometers
scale_factor = 4.0  # to adjust the height of the anomalies for better visualization

# Loop through all times

for jt in range(len(time)):
    print(str(jt) + " / " + str(len(time)))

    output_dir = "output"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    fileOut=os.path.join(output_dir, f"Z500_{np.datetime_as_string(time.values[jt],
     unit='m').replace('T', '_').replace(':', '')}.png")
    
    # Only perform the operations if the output file does not already exist.

    if os.path.exists(fileOut):
        print(f"File {fileOut} already exists, skipping.")
        continue
    else:
        print(f"Creating file {fileOut}...")

        fig = plt.figure(figsize=(10, 10))
        # Modify the view angle by some phase "shift" to make the plot more dynamic

        ax = fig.add_subplot(111, projection='3d')
        ax.view_init(elev=20, azim=jt * 0.1 - 90)  # Modify the view angle by some phase "shift"

        # # White Earth sphere to hide the far-side coastlines
        R_earth = 0.5

        lon_sphere, lat_sphere = np.meshgrid(
            np.linspace(0, 360, 360),
            np.linspace(-90, 90, 180)
        )

        Xe = R_earth * np.cos(np.radians(lat_sphere)) * np.cos(np.radians(lon_sphere))
        Ye = R_earth * np.cos(np.radians(lat_sphere)) * np.sin(np.radians(lon_sphere))
        Ze = R_earth * np.sin(np.radians(lat_sphere))

        #ax.plot_surface(
        #    Xe, Ye, Ze,
        #    color='white',
        #    linewidth=0,
        #    antialiased=False,
        #    shade=False,
        #    alpha=1.0
        #)


        # Plot field of geopotential 
        X = (R + scale_factor * Z500_normalized.isel(valid_time=jt)) * \
        np.cos(np.radians(lat_grid)) * np.cos(np.radians(lon_grid))
        Y = (R + scale_factor * Z500_normalized.isel(valid_time=jt)) * \
        np.cos(np.radians(lat_grid)) * np.sin(np.radians(lon_grid))
        Z = (R + scale_factor * Z500_normalized.isel(valid_time=jt)) * \
        np.sin(np.radians(lat_grid))   # Add the normalized height to the Z coordinate
        # Plot the surface and make sure the colors span the colorbar range.
        # Dont plot edges. Make the surface transparent.



        from matplotlib.colors import Normalize

        # Saturate colors around typical anomalies
        vmax = 0.03   # to adjust depth
        norm = Normalize(vmin=-vmax, vmax=vmax)

        colors = plt.cm.RdBu_r(
        norm(Z500_normalized.isel(valid_time=jt))
        )
        # Plot the surface and color the faces so that the colors span the colorbar range. 
        # Dont plot edges. Make the surface transparent.
        surf = ax.plot_surface(
        X, Y, Z,
        facecolors=colors,
        rstride=1,
        cstride=1,
        antialiased=False,
        shade=False,
        edgecolor='none',
        alpha=0.5
        )

        # Set the axes limits
        ax.set_xlim(-R, R)
        ax.set_ylim(-R, R)
        ax.set_zlim(-R, R)
        ax.set_box_aspect([1,1,0.9])


        # ------------------------------------------------------------
        # Coastlines: Camera direction vector from current azimuth/elevation
        # ------------------------------------------------------------
        import cartopy.feature as cfeature
        import cartopy.io.shapereader as shpreader
        coastline_feature = cfeature.NaturalEarthFeature(
            category='physical',
            name='coastline',
            scale='110m'
        )

        geoms = list(coastline_feature.geometries())
        azim = np.radians(ax.azim)
        elev = np.radians(ax.elev)

        def lonlat_to_xyz(lon, lat, R=1.0):
            lon = np.radians(lon)
            lat = np.radians(lat)

            x = R * np.cos(lat) * np.cos(lon)
            y = R * np.cos(lat) * np.sin(lon)
            z = R * np.sin(lat)
            return x, y, z

        cam_x = np.cos(elev) * np.cos(azim)
        cam_y = np.cos(elev) * np.sin(azim)
        cam_z = np.sin(elev)

        camera_vec = np.array([cam_x, cam_y, cam_z])

        R_coast = 0.95

        for geom in geoms:

            def plot_visible_line(lons, lats):

                x, y, z = lonlat_to_xyz(lons, lats, R_coast)

                # Dot product with camera direction
                visible = (
                    x * camera_vec[0]
                    + y * camera_vec[1]
                    + z * camera_vec[2]
                ) > 0

                # Mask invisible points
                x = np.where(visible, x, np.nan)
                y = np.where(visible, y, np.nan)
                z = np.where(visible, z, np.nan)

                ax.plot(x, y, z, color='black', linewidth=2)

            if geom.geom_type == "LineString":

                lons, lats = geom.xy
                plot_visible_line(lons, lats)

            elif geom.geom_type == "MultiLineString":

                for line in geom.geoms:

                    lons, lats = line.xy
                    plot_visible_line(lons, lats)

    # Titre en français et date sous forme "DD/MM/YYYY à HH:MM"
        date_str = np.datetime_as_string(time.values[jt], unit='m').replace('T', ' à ')
        # Extraire YYYY, MM, DD, HH et mm pour les afficher dans le titre
        YYYY = date_str[0:4]
        MM = date_str[5:7]
        DD = date_str[8:10]
        HH = date_str[13:15]
        mm = date_str[16:18]
    
        date_str = f"{DD}/{MM}/{YYYY} à {HH}:{mm}"

        ax.set_title(f"Hauteur de la couche de pression 500 hPa: {date_str}", fontsize=16)

        # Remove axes labels and ticks
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_zlabel('')
    
        # Add "Animation by F. Massonnet" in the bottom right corner
        ax.text2D(0.95, 0.05, "Animation by F. Massonnet\nData: ERA5 reanalysis", transform=ax.transAxes,
                fontsize=10, color='gray', ha='right')
        
        plt.tight_layout()

        # Save figure with date tag as YYYYMMDDHHMM
        
        os.makedirs(output_dir, exist_ok=True)
        fileOut=os.path.join(output_dir, f"Z500_{np.datetime_as_string(time.values[jt],
        unit='m').replace('T', '_').replace(':', '')}.png")
        plt.savefig(fileOut, dpi=300)
        plt.close()


0 / 616
Creating file output/Z500_2026-05-01_0000.png...
1 / 616
Creating file output/Z500_2026-05-01_0100.png...
2 / 616
Creating file output/Z500_2026-05-01_0200.png...
3 / 616
Creating file output/Z500_2026-05-01_0300.png...
4 / 616
Creating file output/Z500_2026-05-01_0400.png...
5 / 616
Creating file output/Z500_2026-05-01_0500.png...
6 / 616
Creating file output/Z500_2026-05-01_0600.png...
7 / 616
Creating file output/Z500_2026-05-01_0700.png...
8 / 616
Creating file output/Z500_2026-05-01_0800.png...
9 / 616
Creating file output/Z500_2026-05-01_0900.png...
10 / 616
Creating file output/Z500_2026-05-01_1000.png...
11 / 616
Creating file output/Z500_2026-05-01_1100.png...
12 / 616
Creating file output/Z500_2026-05-01_1200.png...
13 / 616
Creating file output/Z500_2026-05-01_1300.png...
14 / 616
Creating file output/Z500_2026-05-01_1400.png...
15 / 616
Creating file output/Z500_2026-05-01_1500.png...
16 / 616
Creating file output/Z500_2026-05-01_1600.png...
17 / 616
Creating file o